# SageMaker + YOLO (Ultralytics) training

`deploy/` provides a **PyTorch + CUDA 12.1** image (`deploy/Dockerfile`) and `scripts/train_yolo.py` is the training entry point. Use this notebook to launch jobs on **ml.g6.xlarge** (NVIDIA L4) or similar.

1. Build and push the image to **ECR** (or use a pre-built PyTorch + Ultralytics image).
2. Upload `data/processed/splits/` (with `data.yaml` using `path: .`) to **S3**.
3. Run an `Estimator` (custom image) with ClearML env vars from `.env` (optional).
4. In the container, run: `python scripts/train_yolo.py --data-yaml <path> --epochs 100 --batch 16 --workers 4 --model-cfg yolov8m.pt`.

In [ ]:
%pip install sagemaker boto3 python-dotenv -q

import os
import sagemaker
from sagemaker.estimator import Estimator
from dotenv import load_dotenv

# Load secrets from a local .env file.
# Ensure this file is never committed to Git.
load_dotenv('../.env')

print(f"SageMaker Version: {sagemaker.__version__}")

### 1. Configure role, ECR image, and environment

- **Instance**: `ml.g6.xlarge` (1x L4, 16 GB system RAM) matches repo defaults: `batch=16`, `workers=4`.
- **Data**: S3 should mirror `data/processed/splits/` so `data.yaml` resolves (`path: .` is relative to that folder).
- **ClearML**: load keys from `../.env` and pass via `environment=` on the Estimator.

In [ ]:
# 1) Role: replace with your account, or in Studio: sagemaker.get_execution_role()
role = "arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-20230101T000000"

# 2) ECR image built from PROJECTS/CrowdNav/deploy/Dockerfile (PyTorch + requirements.txt)
image_uri = "123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/crowdnav-yolo:latest"

# 3) Custom container Estimator — training command is your train script inside the image
#    (or use a wrapper script as container ENTRYPOINT for SageMaker).
yolo_estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.g6.xlarge",  # L4: matches --batch 16 / --workers 4 defaults
    output_path="s3://my-crowdnav-bucket/models/",
    environment={
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
        "CLEARML_API_ACCESS_KEY": os.environ.get("CLEARML_API_ACCESS_KEY", ""),
        "CLEARML_API_SECRET_KEY": os.environ.get("CLEARML_API_SECRET_KEY", ""),
    },
)

### 2. Fit: S3 data channel

SageMaker downloads the channel to `SM_CHANNEL_TRAINING` (e.g. `/opt/ml/input/data/training/`). After sync, `data-yaml` should be that path + `splits/data.yaml` (or your layout). For a custom image, set the container `ENTRYPOINT` to `python scripts/train_yolo.py` with the right `--data-yaml` or use `PyTorch` estimator with `source_dir` + `entry_point` instead of a raw `Estimator`.

In [ ]:
# S3 prefix should contain splits/data.yaml and train/val/test (same layout as data/processed/splits/).
# Uncomment when ECR image and bucket are ready:
# yolo_estimator.fit({"training": "s3://my-bucket/crowdnav/splits/"})
# In the training container, e.g.:
#   python scripts/train_yolo.py --data-yaml /opt/ml/input/data/training/data.yaml --epochs 100 --workers 4